# 4) Probabilistic forecasts: intervals, quantiles and calibration

## Objectives

At the end of this notebook you should be able to:

- turn a point forecast into a **prediction interval** from past forecast errors — and choose those errors honestly
- check **calibration** (does an 80 % interval contain the truth 80 % of the time?) overall and **conditionally**, by season and hour
- train **quantile** models, recognise when they are overconfident, and fix them with **conformal calibration**
- score probabilistic forecasts with the **pinball loss**, which rewards calibration and sharpness together
- forecast **solar**, whose uncertainty vanishes at night and peaks on bright but uncertain days

A point forecast of 2,400 MW answers the wrong question for most decisions. A TSO sizing reserves, a trader bidding a portfolio, a battery operator planning tomorrow — all need to know **how wrong** the forecast may be: is 2,550 MW plausible, is 2,700? This lesson attaches honest uncertainty to lesson 2's demand forecast, then turns to solar.

In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingRegressor

from forecast_data import load_data, solar_data, temperature_forecast

warnings.filterwarnings("ignore")
logging.getLogger("sklearn").setLevel(logging.ERROR)

### Intervals from past errors — which errors?

The simplest probabilistic forecast adds the **quantiles of past errors** to the point forecast: if 10 % of past errors were below −120 MW and 10 % above +118 MW, the 80 % interval is the forecast −120 to +118 MW. Everything depends on **which** past errors. Two candidates:

- the **in-sample** residuals of a model fitted on 2023, computed with the actual temperature — what a regression summary reports
- the errors the 2023 model made **forecasting 2024**, with the temperature forecast — a genuine out-of-sample test of the same forecasting process

Both intervals are attached to the 2025 forecast of the model fitted on 2023–2024 (lesson 2's model) and checked against 2025.

In [2]:
df = load_data()
df["temperature_forecast"] = temperature_forecast(df)
idx, dow = df.index, df.index.dayofweek
df["daytype"] = np.where(df.holiday | (dow == 6), "sunday_or_holiday", np.where(dow == 5, "saturday", "workday"))
df["hour"], df["month"] = idx.hour, idx.month
FORMULA = "demand ~ C(hour):C(daytype) + C(month) + bs(temperature, df=6, lower_bound=-12, upper_bound=40)"
honest = df.assign(temperature=df.temperature_forecast)                 # inputs as known at noon on D-1

m2023 = smf.ols(FORMULA, df.loc["2023"]).fit()
errors = {"in-sample 2023, actual temperature": (df.demand - m2023.predict(df)).loc["2023"],
          "forecasting 2024, temperature forecast": (df.demand - m2023.predict(honest)).loc["2024"]}
model = smf.ols(FORMULA, df.loc[:"2024"]).fit()
point = model.predict(honest).loc["2025"]
actual = df.demand.loc["2025"]


def coverage(lower, upper, y=actual):
    return float(((y >= lower) & (y <= upper)).mean())


rows = {}
for name, e in errors.items():
    lo, hi = np.quantile(e, [0.1, 0.9])
    rows[name] = {"interval_low_MW": lo, "interval_high_MW": hi, "width_MW": hi - lo,
                  "coverage_2025": coverage(point + lo, point + hi)}
pd.DataFrame(rows).T.round(3)

                                        interval_low_MW  interval_high_MW  \
in-sample 2023, actual temperature              -96.096            97.538   
forecasting 2024, temperature forecast         -128.663           108.873   

                                        width_MW  coverage_2025  
in-sample 2023, actual temperature       193.634          0.714  
forecasting 2024, temperature forecast   237.536          0.806  

The in-sample residuals give an interval 194 MW wide that holds the truth in only **71 %** of 2025's hours — not the 80 % it promises. The errors of a genuine forecast of 2024 give an interval 238 MW wide that covers **80.6 %**. The difference is lessons 1 and 2 again, now applied to uncertainty: the in-sample residuals were computed with the **actual** temperature, so they leave out the error of the temperature forecast, and they measure how well the model fits data it has seen, not how well it forecasts data it has not. An interval can only be as honest as the errors it is built from: take them from a forecast made exactly as it will be made in operation.

### Calibrated on average — and in detail?

In [3]:
lo, hi = np.quantile(errors["forecasting 2024, temperature forecast"], [0.1, 0.9])
inside = (actual >= point + lo) & (actual <= point + hi)
by_month = inside.groupby(actual.index.month).mean()
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(by_month.index, by_month.to_numpy(), color="tab:blue")
ax.axhline(0.8, color="black", ls="--", lw=1, label="target 80 %")
ax.set(xlabel="month of 2025", ylabel="share of hours inside", ylim=(0.5, 1), xticks=range(1, 13),
       title="Coverage of the 80 % interval by month")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f"overall {inside.mean():.3f}; by month from {by_month.min():.2f} to {by_month.max():.2f}")

overall 0.806; by month from 0.72 to 0.87


Over the year the interval is calibrated: 80.6 % against the 80 % promised. Month by month it is not — from 72 % in June to 87 % in January and July. Coverage is too low in some months and too high in others, and the two cancel in the annual figure — the same trap as lesson 1's bias, which looked small over the year while large seasonal biases cancelled. For a TSO the months where coverage falls short are the ones that matter: an interval used to size reserves that is too narrow in June means too little reserve held in June. Always check calibration conditionally — by season, by hour, by type of day, and above all in the conditions the forecast is used for.

### Quantile models, and conformal calibration

Instead of one interval for all hours, a **quantile model** predicts the 10 % and 90 % quantiles directly from the features, so the interval can widen when the weather is uncertain. Gradient boosting does this with `loss="quantile"`. To keep 2024 for calibration, the quantile models are fitted on 2023 only:

In [4]:
dt = np.where(df.holiday | (dow == 6), 2, np.where(dow == 5, 1, 0))
X = pd.DataFrame({"hour": idx.hour, "daytype": dt, "day_of_year": idx.dayofyear,
                  "temperature_forecast": df.temperature_forecast}, index=idx)
y = df.demand
fit, calib, test = idx.year == 2023, idx.year == 2024, idx.year == 2025


def quantile_model(alpha):
    return HistGradientBoostingRegressor(loss="quantile", quantile=alpha, max_iter=600, learning_rate=0.05,
                                         random_state=0).fit(X[fit], y[fit])


q10, q90 = quantile_model(0.1), quantile_model(0.9)
raw_lo, raw_hi = q10.predict(X[test]), q90.predict(X[test])
print(f"quantile boosting, raw: coverage {coverage(raw_lo, raw_hi):.3f}, mean width {np.mean(raw_hi - raw_lo):.0f} MW")
print(f"on its own training year it covers {float(((y[fit] >= q10.predict(X[fit])) & (y[fit] <= q90.predict(X[fit]))).mean()):.3f}")

quantile boosting, raw: coverage 0.494, mean width 134 MW


on its own training year it covers 0.798


The model is badly **overconfident**: its "80 %" interval holds the truth in only about half of 2025's hours, because it learned the spread of its errors on the data it was fitted to — and errors on new data are larger. **Conformal calibration** repairs this with one number. On the calibration year, measure how far each actual value lies outside the interval (negative if inside): s = max(lower − y, y − upper). Then widen both ends by the 80 % quantile of these scores. If the calibration year resembles the future, the widened interval covers 80 %:

In [5]:
scores = np.maximum(q10.predict(X[calib]) - y[calib], y[calib] - q90.predict(X[calib]))
n = len(scores)
margin = np.quantile(scores, np.ceil((n + 1) * 0.8) / n)
cqr_lo, cqr_hi = raw_lo - margin, raw_hi + margin
print(f"conformal margin {margin:.1f} MW -> coverage {coverage(cqr_lo, cqr_hi):.3f}, mean width {np.mean(cqr_hi - cqr_lo):.0f} MW")

conformal margin 62.0 MW -> coverage 0.794, mean width 258 MW


Calibrated on 2024, the boosting quantiles widen by 62 MW on each side and cover **79.4 %** of 2025 — the coverage they promise. The raw model's overconfidence is clear in the numbers: it covered 79.8 % of its own training year but only 49.4 % of 2025. Conformal calibration needs no assumption about the shape of the errors; it assumes only that the calibration year is representative of the future. That is also its limit: after a change the calibration year has not seen — a new kind of demand, a heat wave beyond anything in the data — the margin is no longer right.

### One score for the whole distribution: the pinball loss

Coverage alone can be gamed: an interval from 0 to 10,000 MW covers 100 %. A good probabilistic forecast must be **calibrated** and **sharp** (narrow). The **pinball loss** of a forecast q of the α-quantile scores both at once: α · (y − q) if the truth is above it, (1 − α) · (q − y) if below. It is minimised by the true quantile — so lower is better, and neither a too-wide nor a too-narrow interval can win.

In [6]:
def pinball(y, q, alpha):
    d = np.asarray(y) - np.asarray(q)
    return float(np.mean(np.maximum(alpha * d, (alpha - 1) * d)))


methods = {"errors of 2024 (constant width)": (point + lo, point + hi),
           "quantile boosting, raw": (raw_lo, raw_hi),
           "quantile boosting + conformal": (cqr_lo, cqr_hi),
           "absurdly wide: 0 to 10,000 MW": (np.zeros(len(actual)), np.full(len(actual), 10_000.0))}
table = pd.DataFrame({name: {"coverage": coverage(l, h), "mean_width_MW": float(np.mean(h - l)),
                             "pinball_q10": pinball(actual, l, 0.1), "pinball_q90": pinball(actual, h, 0.9)}
                      for name, (l, h) in methods.items()}).T
table["pinball_mean"] = table[["pinball_q10", "pinball_q90"]].mean(axis=1)
table.round(3)

                                 coverage  mean_width_MW  pinball_q10  \
errors of 2024 (constant width)     0.806        237.536       17.476   
quantile boosting, raw              0.494        133.875       19.548   
quantile boosting + conformal       0.794        257.778       17.736   
absurdly wide: 0 to 10,000 MW       1.000      10000.000      204.084   

                                 pinball_q90  pinball_mean  
errors of 2024 (constant width)       15.924        16.700  
quantile boosting, raw                25.028        22.288  
quantile boosting + conformal         18.050        17.893  
absurdly wide: 0 to 10,000 MW        795.916       500.000  

The pinball loss ranks the methods sensibly. The absurdly wide interval covers everything and scores terribly — 500 against about 17 — because it is useless. The raw quantile model is punished for being overconfident (22.3). And between the two calibrated intervals, the **simple** one wins: the constant width from 2024's errors scores 16.7, the conformal boosting quantiles 17.9, because they are calibrated but wider (258 against 238 MW). A flexible quantile model pays off when the uncertainty really changes with the conditions; in these data the size of demand's error changes only moderately with season and weather, and the extra flexibility costs more in noise than it gains. As in lesson 3: the more complex method has to earn its place against an honest simple one.

### Solar: uncertainty that follows the sun

In [7]:
sol = solar_data(df)
s, f = sol.solar.loc["2025"], sol.solar_forecast.loc["2025"]
print(f"solar fleet 1,500 MW: {s.sum() / 1e6:.2f} TWh in 2025, capacity factor {s.mean() / 1500:.1%}")
daylight = s > 1
err = f - s
low = daylight & (s < 20)
print(f"forecast MAE: all hours {err.abs().mean():.1f} MW, daylight hours {err[daylight].abs().mean():.1f} MW")
print(f"MAPE in daylight {100 * (err.abs() / s)[daylight].mean():.0f} %, in the {low.sum()} hours below 20 MW "
      f"{100 * (err.abs() / s)[low].mean():.0f} %; at night undefined (0 / 0)")

solar fleet 1,500 MW: 1.77 TWh in 2025, capacity factor 13.5%
forecast MAE: all hours 35.7 MW, daylight hours 71.5 MW
MAPE in daylight 30 %, in the 180 hours below 20 MW 82 %; at night undefined (0 / 0)


Percentage errors break down for solar: they are undefined at night and explode at dawn and dusk, when a 10 MW miss on 12 MW of output is an error of more than 80 % that matters to no one. Report solar errors in MW, or relative to **capacity**.

The uncertainty itself is not constant either. At night there is none; at noon on a summer day it can be several hundred MW. A constant-width interval would be absurd — it would allow negative output at night. Instead, take the errors of **2024** separately for each **level of the forecast**, and clip the interval to what the fleet can physically produce, 0 to 1,500 MW:

In [8]:
s24, f24 = sol.solar.loc["2024"], sol.solar_forecast.loc["2024"]
bins = [-1, 1, 100, 300, 600, 900, 1500]
level24 = pd.cut(f24, bins)
q = (s24 - f24).groupby(level24, observed=True).quantile([0.1, 0.9]).unstack()
level = pd.cut(f, bins)
lo_s = (f + q.loc[level, 0.1].to_numpy()).clip(0, 1500)
hi_s = (f + q.loc[level, 0.9].to_numpy()).clip(0, 1500)
inside_s = (s >= lo_s) & (s <= hi_s)
by_level = pd.DataFrame({"hours": level.value_counts().sort_index(),
                         "mean_width_MW": (hi_s - lo_s).groupby(level, observed=True).mean(),
                         "coverage": inside_s.groupby(level, observed=True).mean()})
print(f"overall coverage {inside_s.mean():.3f}; in daylight {inside_s[daylight].mean():.3f}")
by_level.round(3)

overall coverage 0.889; in daylight 0.778


                hours  mean_width_MW  coverage
solar_forecast                                
(-1, 1]          4386          0.000     0.999
(1, 100]          910         81.534     0.787
(100, 300]       1206        151.096     0.767
(300, 600]       1078        255.156     0.751
(600, 900]        734        368.292     0.815
(900, 1500]       446        290.005     0.798

In [9]:
fig, ax = plt.subplots(figsize=(9, 3.4))
week = slice("2025-06-02", "2025-06-08 23:00")
ax.fill_between(lo_s.loc[week].index, lo_s.loc[week], hi_s.loc[week], alpha=0.3, label="80 % interval")
ax.plot(f.loc[week], lw=1, label="forecast")
ax.plot(s.loc[week], color="black", lw=1.2, label="actual")
ax.set(ylabel="MW", ylim=(0, 1650), title="Solar, one June week of 2025: the interval follows the sun")
ax.legend(fontsize=8, ncol=3, loc="upper center")
plt.tight_layout()
plt.show()

Solar's uncertainty is not one number. The interval has zero width at night, about 80 MW when little output is forecast, and more than 350 MW when the forecast is between 600 and 900 MW — plausibly because such hours belong to days expected to be bright but not cloudless, where a few clouds more or less change the output most. Above 900 MW it narrows again, probably because such a forecast means a clear day is expected and the fleet cannot produce much more than that, only less.

Two warnings about the headline numbers. The overall coverage of 88.9 % flatters: the 4,386 night hours are covered trivially by a zero-width interval around zero output, and in **daylight** coverage is 77.8 %, a little short of the 80 % promised. The same trick works for point forecasts — the MAE over all hours (35.7 MW) is half the MAE over daylight hours (71.5 MW). Report solar accuracy for the hours in which the sun shines, and in MW or as a share of capacity, never as MAPE. The figure also shows how solar forecasts fail: on 2 June the forecast expected a cloudier day than came, and the actual output lies above the interval for most of the day. A day's cloudiness is one error shared by all its hours, so solar misses come in whole days, not in single hours.

## Check your understanding

**Calibration**

1. Why did the interval built from in-sample residuals cover only about 71 % of 2025? Name both missing sources of error.
2. The interval is calibrated over the year but not in every month. Why would a TSO care about the months in which coverage is lowest?

**Quantiles and scores**

1. Explain in two sentences what conformal calibration assumes about the calibration year, and when that assumption would fail.
2. Why does the pinball loss punish both the overconfident and the absurdly wide interval, while coverage punishes only one of them?